# Task 3: Car Price Prediction with ML

**Author**: Pratham Bhat  
**Objective**: Predict car selling prices using Linear Regression and Random Forest  
**Dataset**: Car price data (user-provided CSV)  
**Models**: Linear Regression, Random Forest Regressor

## 1. Load Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [ ]:
# COLUMN NAME CONFIG - Edit if your CSV has different column names
SELLING_PRICE_COLUMN = 'selling_price'
PRESENT_PRICE_COLUMN = 'present_price'
KMS_DRIVEN_COLUMN = 'kms_driven'
FUEL_TYPE_COLUMN = 'fuel_type'
SELLER_TYPE_COLUMN = 'seller_type'
TRANSMISSION_COLUMN = 'transmission'
OWNER_COLUMN = 'owner'
YEAR_COLUMN = 'year'

# Load data
df = pd.read_csv('car_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nInfo:")
print(df.info())

## 2. Data Preprocessing

In [ ]:
# Drop missing values
df = df.dropna(subset=[SELLING_PRICE_COLUMN, PRESENT_PRICE_COLUMN, KMS_DRIVEN_COLUMN])

# Encode categorical variables
df_encoded = df.copy()
categorical_cols = [FUEL_TYPE_COLUMN, SELLER_TYPE_COLUMN, TRANSMISSION_COLUMN]
for col in categorical_cols:
    if col in df_encoded.columns:
        df_encoded[col] = pd.Categorical(df_encoded[col]).codes

print(f"Preprocessed shape: {df_encoded.shape}")
print(f"\nPrice Statistics:")
print(df[SELLING_PRICE_COLUMN].describe())

## 3. Exploratory Data Analysis

In [ ]:
# Correlation heatmap
numeric_cols = [SELLING_PRICE_COLUMN, PRESENT_PRICE_COLUMN, KMS_DRIVEN_COLUMN, YEAR_COLUMN]
numeric_cols = [col for col in numeric_cols if col in df.columns]

fig, ax = plt.subplots(figsize=(10, 8))
correlation = df[numeric_cols].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, ax=ax, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
ax.set_title('Car Features - Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Selling Price vs Features', fontsize=14, fontweight='bold')

feature_pairs = [(PRESENT_PRICE_COLUMN, 'Present Price'),
                 (KMS_DRIVEN_COLUMN, 'KMs Driven'),
                 (YEAR_COLUMN, 'Year'),
                 ('owner', 'Owner')]

for idx, (feature, label) in enumerate(feature_pairs):
    if feature in df.columns:
        ax = axes[idx // 2, idx % 2]
        ax.scatter(df[feature], df[SELLING_PRICE_COLUMN], alpha=0.5, s=50, edgecolors='navy')
        ax.set_xlabel(label, fontsize=11, fontweight='bold')
        ax.set_ylabel('Selling Price', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Model Training

In [ ]:
# Prepare features and target
feature_cols = [PRESENT_PRICE_COLUMN, KMS_DRIVEN_COLUMN, FUEL_TYPE_COLUMN, 
                SELLER_TYPE_COLUMN, TRANSMISSION_COLUMN, OWNER_COLUMN, YEAR_COLUMN]
feature_cols = [col for col in feature_cols if col in df_encoded.columns]

X = df_encoded[feature_cols]
y = df_encoded[SELLING_PRICE_COLUMN]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

In [ ]:
# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_r2 = r2_score(y_test, y_pred_lr)

print(f"Linear Regression:")
print(f"  MAE: {lr_mae:.4f}")
print(f"  R²: {lr_r2:.4f}")

In [ ]:
# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)

print(f"Random Forest:")
print(f"  MAE: {rf_mae:.4f}")
print(f"  R²: {rf_r2:.4f}")

## 5. Model Evaluation

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_lr, alpha=0.6, s=50, edgecolors='navy')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Price', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Predicted Price', fontsize=11, fontweight='bold')
axes[0].set_title(f'Linear Regression (R²={lr_r2:.4f})', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_test, y_pred_rf, alpha=0.6, s=50, color='coral', edgecolors='darkred')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[1].set_xlabel('Actual Price', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Predicted Price', fontsize=11, fontweight='bold')
axes[1].set_title(f'Random Forest (R²={rf_r2:.4f})', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], 
       color='steelblue', alpha=0.8, edgecolor='navy')
ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
ax.set_title('Feature Importance - Random Forest', fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 6. Summary

**Best Model**: Random Forest (typically better for non-linear relationships)  
**R² Score**: Indicates model fit quality  
**Feature Importance**: Shows which features most impact price